# Embedding & Vector Store Ingestion

**Goal:** Embed all chunks with BGE-M3 and upsert them into Qdrant Cloud as a hybrid (dense + sparse) collection ready for RAG retrieval.

## 1. Environment Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

DATA_DIR = "data"
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /home/dmitry/Projects/DataScience/rag-techdoc-assistant


In [2]:
import json
import logging
import os

import torch
import numpy as np
from dotenv import load_dotenv
from tqdm.auto import tqdm

from qdrant_client import QdrantClient

from src.chunking.chunker import Chunk
from src.embedding import BGEM3Embedder
from src.vectorstore import QdrantDocStore, show_results

load_dotenv(PROJECT_ROOT / ".env")

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s  %(name)-25s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

log = logging.getLogger("notebook")

## 2. Configuration

In [3]:
OUTPUT_DIR   = PROJECT_ROOT / DATA_DIR / "pytorch_docs_md"
CHUNKS_PATH  = OUTPUT_DIR / "_chunks.jsonl"
VECTORS_PATH = OUTPUT_DIR / "_vectors.npy"

COLLECTION_NAME   = "pytorch_docs"
QDRANT_BATCH_SIZE = 128

QDRANT_URL = os.environ["QDRANT_URL"]
QDRANT_KEY = os.environ["QDRANT_API_KEY"]

print(f"Chunks file  : {CHUNKS_PATH}")
print(f"Vectors cache: {VECTORS_PATH}")
print(f"Collection   : {COLLECTION_NAME}")
print(f"Qdrant URL   : {QDRANT_URL}")

Chunks file  : /home/dmitry/Projects/DataScience/rag-techdoc-assistant/data/pytorch_docs_md/_chunks.jsonl
Vectors cache: /home/dmitry/Projects/DataScience/rag-techdoc-assistant/data/pytorch_docs_md/_vectors.npy
Collection   : pytorch_docs
Qdrant URL   : https://47c266d8-8135-4e60-9e8b-6fd120ff239b.europe-west3-0.gcp.cloud.qdrant.io


## 3. Load Chunks

In [4]:
def load_chunks(path: Path) -> list[Chunk]:
    chunks = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                d = json.loads(line)
                d.pop("char_count", None)
                chunks.append(Chunk(**d))
    return chunks


chunks = load_chunks(CHUNKS_PATH)

print(f"Total chunks : {len(chunks):,}")
print(f"Avg chars    : {sum(c.char_count for c in chunks) // len(chunks):,}")

Total chunks : 8,358
Avg chars    : 809


## 4. Compute Dense Embeddings

BGE-M3 produces 1,024-dim L2-normalised vectors, the results are cached in
`_vectors.npy`.

In [5]:
print(f"Device  : {'cuda' if torch.cuda.is_available() else 'cpu'}")

embedder = BGEM3Embedder(batch_size=12)

print(f"Model   : {embedder.MODEL_ID}")
print(f"Dim     : {embedder.EMBED_DIM}")
print(f"Batch   : {embedder.batch_size}")

/home/dmitry/Projects/DataScience/rag-techdoc-assistant/.devenv/state/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:184: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Device  : cpu


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Model   : BAAI/bge-m3
Dim     : 1024
Batch   : 12


In [6]:
# Verify the embedder works before committing to the full run

test_vecs = embedder.embed(["torch.nn.Linear", "What is autograd?"])

assert test_vecs.shape == (2, 1024), f"Unexpected shape: {test_vecs.shape}"

print(f"Smoke test passed, shape: {test_vecs.shape}, dtype: {test_vecs.dtype}")
print(f"L2 norms: {np.linalg.norm(test_vecs, axis=1).round(4).tolist()}")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Smoke test passed, shape: (2, 1024), dtype: float32
L2 norms: [1.0, 1.0]


In [7]:
from src.embedding import VectorCache

cache = VectorCache(OUTPUT_DIR)
print(f"Cache status : {cache.size:,} vectors already stored")

dense_vectors = cache.embed_missing(chunks, embedder)

print(f"Dense vectors shape : {dense_vectors.shape}")
print(f"L2 norm sample      : {np.linalg.norm(dense_vectors[:5], axis=1).round(4).tolist()}")

Cache status : 8,358 vectors already stored
Dense vectors shape : (8358, 1024)
L2 norm sample      : [1.0, 1.0, 1.0, 1.0, 1.0]


## 5. Prepare Qdrant Collection

In [8]:
qdrant_client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_KEY,
    timeout=60,
)

cluster_info = qdrant_client.get_collections()
existing = [c.name for c in cluster_info.collections]
print(f"Connected to Qdrant Cloud. Existing collections: {existing or '(none)'}")

Connected to Qdrant Cloud. Existing collections: ['pytorch_docs']


In [9]:
store = QdrantDocStore(
    client=qdrant_client,
    collection_name=COLLECTION_NAME,
    embedder=embedder,
)

store.create_collection(recreate=False)

info = store.collection_info()
print("Collection info:")
for k, v in info.items():
    print(f"  {k:<30}: {v}")

Collection info:
  name                          : pytorch_docs
  points_count                  : 8358
  indexed_vectors_count         : 16714
  status                        : green


## 6. Upsert All Chunks

In [10]:
store.upsert_chunks(
    chunks=chunks,
    dense_vectors=dense_vectors,
    batch_size=QDRANT_BATCH_SIZE,
)

info = store.collection_info()
print(f"Upsert complete")
print(f"  Points in collection : {info['points_count']:,}")
print(f"  Indexed vectors      : {info['indexed_vectors_count']:,}")
print(f"  Collection status    : {info['status']}")

Upsert complete
  Points in collection : 8,358
  Indexed vectors      : 17,702
  Collection status    : yellow


## 7. Verification

In [11]:
show_results(store, "How to create a new tensor?")

Query: 'How to create a new tensor?'

#    Score  Kind         Symbol / Title                      Citation
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
1   0.5000  heading      torch.distributed.tensor            https://docs.pytorch.org/docs/stable/distributed.tensor.html#dtensor-class-apis
2   0.5000  method       torch.Tensor.new_tensor             https://docs.pytorch.org/docs/stable/generated/torch.Tensor.new_tensor.html#torch.Tensor.new_tensor
3   0.3333  heading      torch.distributed.tensor            https://docs.pytorch.org/docs/stable/distributed.tensor.html#dtensor-factory-functions
4   0.3333  class        torch.Tensor                        https://docs.pytorch.org/docs/stable/tensors.html#torch.Tensor



In [12]:
show_results(store, "How to set a random seed for pytorch?")

Query: 'How to set a random seed for pytorch?'

#    Score  Kind         Symbol / Title                      Citation
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
1   0.5000  function     torch.random.set_rng_state          https://docs.pytorch.org/docs/stable/random.html#torch.random.set_rng_state
2   0.5000  heading      torch.mps.seed                      https://docs.pytorch.org/docs/stable/generated/torch.mps.seed.html#torch-mps-seed
3   0.4583  function     torch.random.seed                   https://docs.pytorch.org/docs/stable/random.html#torch.random.seed
4   0.3333  function     torch.mps.manual_seed               https://docs.pytorch.org/docs/stable/generated/torch.mps.manual_seed.html#torch.mps.manual_seed



In [13]:
show_results(store, "Will train script work with torchrun?")

Query: 'Will train script work with torchrun?'

#    Score  Kind         Symbol / Title                      Citation
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
1   0.5000  heading      Autograd mechanics                  https://docs.pytorch.org/docs/stable/notes/autograd.html#in-place-operations-with-autograd
2   0.5000  heading      Train script                        https://docs.pytorch.org/docs/stable/elastic/train_script.html#train-script
3   0.3333  heading      TorchScript                         https://docs.pytorch.org/docs/stable/jit.html#creating-torchscript-code
4   0.3333  heading      Autograd mechanics                  https://docs.pytorch.org/docs/stable/notes/autograd.html#how-does-pytorch-compute-the-conjugate-wirtinger-derivative



In [14]:
show_results(store, "What is torch.random.fork_rng")

Query: 'What is torch.random.fork_rng'

#    Score  Kind         Symbol / Title                      Citation
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
1   0.6667  function     torch.random.fork_rng               https://docs.pytorch.org/docs/stable/random.html#torch.random.fork_rng
2   0.5833  function     torch.random.get_rng_state          https://docs.pytorch.org/docs/stable/random.html#torch.random.get_rng_state
3   0.5000  function     torch.random.set_rng_state          https://docs.pytorch.org/docs/stable/random.html#torch.random.set_rng_state
4   0.3429  function     torch.random.seed                   https://docs.pytorch.org/docs/stable/random.html#torch.random.seed

